# ACIC 2016 Benchmark with AIPyW

This notebook mirrors the Hainmueller simulation pattern in `00_demo.ipynb`, but uses the native Python ACIC 2016 benchmark generator. The generator keeps the original 77 setting by 100 replication layout and returns full potential-outcome truth, so we can compare AIPW estimates against `true_att` or `true_ate`.

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier, LGBMRegressor

from aipyw import AIPyW
from aipyw.benchmarks import (
    ACIC2016_N_REPLICATIONS,
    ACIC2016_N_SETTINGS,
    load_acic2016_parameters,
    simulate_acic2016,
)

np.random.seed(42)
BALANCING_OBJS = ["quadratic", "entropy", "balnet"]

## Parameter Grid

The benchmark has 77 ACIC-style settings. Each row controls the treatment model, baseline treatment share, overlap, response surface, treatment-response alignment, and treatment-effect heterogeneity.

In [ ]:
params = load_acic2016_parameters()
params.head()

In [ ]:
ACIC2016_N_SETTINGS, ACIC2016_N_REPLICATIONS

## One Replication

This helper follows the same shape as the Hainmueller example: generate one dataset, fit AIPyW, and return the scalar treatment-effect estimate plus the known target.

In [ ]:
def one_rep_acic16(setting, replication, riesz_method, target="att", **kwargs):
    sample = simulate_acic2016(setting=setting, replication=replication, standardize=True)
    y, d, X = sample.as_tuple()

    m1 = LGBMRegressor(verbose=-1, n_jobs=1)
    m2 = LGBMClassifier(verbose=-1, n_jobs=1)
    aipw = AIPyW(propensity_model=m2, outcome_model=m1, riesz_method=riesz_method, **kwargs)
    aipw.fit(X, d, y, n_rff=100)

    estimate = aipw.summary()["1 vs 0"]["effect"]
    truth = sample.true_att if target == "att" else sample.true_ate
    return {
        "setting": setting,
        "replication": replication,
        "method": riesz_method if "bal_obj" not in kwargs else f"{riesz_method}:{kwargs['bal_obj']}",
        "target": target,
        "estimate": estimate,
        "truth": truth,
        "error": estimate - truth,
    }

Moderate starting point: the first ACIC16 setting and first replication.

In [ ]:
%%time
one_rep_acic16(1, 1, "ipw")

In [ ]:
%%time
one_rep_acic16(1, 1, "ipw-hajek")

In [ ]:
%%time
one_rep_acic16(1, 1, "linear")

In [ ]:
%%time
pd.DataFrame(
    [one_rep_acic16(1, 1, "balancing", bal_obj=bal_obj) for bal_obj in BALANCING_OBJS]
)

## Small Grid

Run a small slice of the 77 x 100 benchmark grid. Expanding `settings` and `replications` gives the full benchmark.

In [ ]:
def acic16_grid(settings=(1, 2, 3), replications=(1, 2), methods=("ipw", "ipw-hajek", "linear")):
    rows = []
    for setting in settings:
        for replication in replications:
            for method in methods:
                rows.append(one_rep_acic16(setting, replication, method))
    return pd.DataFrame(rows)

results = acic16_grid()
results

In [ ]:
results.assign(abs_error=lambda d: d["error"].abs()).groupby("method").agg(
    n=("error", "size"),
    bias=("error", "mean"),
    mae=("abs_error", "mean"),
    rmse=("error", lambda x: np.sqrt(np.mean(np.square(x)))),
)

## Harder Settings

The parameter grid includes poor overlap and higher treatment-effect heterogeneity. These are useful smoke tests before running the full 77 x 100 sweep.

In [ ]:
params.loc[
    (params["overlap.trt"] != "full") & (params["te.hetero"] == "high"),
    ["model.trt", "root.trt", "overlap.trt", "model.rsp", "alignment", "te.hetero"],
].head()

In [ ]:
hard_settings = params.index[(params["overlap.trt"] != "full") & (params["te.hetero"] == "high")][:3] + 1
acic16_grid(settings=hard_settings, replications=(1,), methods=("ipw", "ipw-hajek", "linear"))